# Day 069 — Exercise 1: Build Extraction Prompt

**What you'll build:** `build_extraction_prompt(schema_cls) -> str` — generate a vision LLM prompt that embeds the Pydantic model's JSON schema.

**Why it matters:** The prompt is the single most important input to the extraction pipeline. Embedding the schema as JSON gives the model precise field names and types to target — dramatically improving the likelihood of a parseable, schema-conforming response.

In [ ]:
import json
from pydantic import BaseModel

class ProductInfo(BaseModel):
    name:     str
    price:    float
    category: str = ''


## Task

Implement `build_extraction_prompt(schema_cls) -> str`:

1. `schema_json = json.dumps(schema_cls.model_json_schema(), indent=2)`
2. Build a prompt string that:
   - Instructs the model to return ONLY valid JSON
   - Embeds `schema_json` in a labeled `Schema:` block
   - Ends with a constraint like "Return ONLY the JSON object"

The checks verify that field names appear in the prompt and the embedded schema is valid JSON.

## Your Implementation

In [ ]:
def build_extraction_prompt(schema_cls) -> str:
    """Build a vision LLM prompt for structured JSON extraction.

    Embeds the Pydantic model's JSON schema in the instruction.
    The prompt must instruct the model to return ONLY a JSON object
    matching the schema.

    Args:
        schema_cls: Pydantic BaseModel subclass
    Returns:
        Prompt string containing the JSON schema and extraction instruction
    """
    raise NotImplementedError


In [ ]:
def build_extraction_prompt(schema_cls) -> str:
    schema_json = json.dumps(schema_cls.model_json_schema(), indent=2)
    return (
        'Extract structured data from this image and return ONLY valid JSON '
        'matching this schema exactly. Do not include any explanation, '
        'markdown, or code blocks.\n\n'
        f'Schema:\n{schema_json}\n\n'
        'Return ONLY the JSON object, nothing else.'
    )


## Automated checks

In [ ]:
score, total = 0, 5
try:
    prompt = build_extraction_prompt(ProductInfo)
    assert isinstance(prompt, str), f"Expected str, got {type(prompt)}"
    score += 1; print("\u2705 returns a string")

    assert 'JSON' in prompt or 'json' in prompt, "Prompt should mention JSON"
    score += 1; print("\u2705 prompt mentions JSON")

    # Schema field names should appear in the prompt
    assert 'name' in prompt, "Field 'name' should appear in prompt"
    assert 'price' in prompt, "Field 'price' should appear in prompt"
    score += 1; print("\u2705 field names (name, price) appear in prompt")

    # The embedded schema should be parseable JSON
    import re
    json_block = re.search(r'\{[\s\S]*\}', prompt)
    assert json_block, "Prompt should contain a JSON schema block"
    parsed = json.loads(json_block.group(0))
    assert isinstance(parsed, dict)
    score += 1; print("\u2705 embedded schema block is valid JSON")

    # Works with a different schema
    class _Other(BaseModel):
        merchant: str
        total: float
    p2 = build_extraction_prompt(_Other)
    assert 'merchant' in p2 and 'total' in p2
    score += 1; print("\u2705 works with a different schema class")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_extraction_prompt(schema_cls) -> str:
    schema_json = json.dumps(schema_cls.model_json_schema(), indent=2)
    return (
        'Extract structured data from this image and return ONLY valid JSON '
        'matching this schema exactly. Do not include any explanation, '
        'markdown, or code blocks.\n\n'
        f'Schema:\n{schema_json}\n\n'
        'Return ONLY the JSON object, nothing else.'
    )
```

**Why `model_json_schema()` and not `model_fields`?** JSON Schema is the standard format LLMs understand — it has type information, required flags, and default values, all in a format the model has seen many times during training. `model_fields` is a Pydantic-internal dict that the model has not seen in training data.

</details>